In [1]:
# Run this in a Jupyter cell to overwrite ~/tisd/tisd_engine_mlx.py
engine_code = """import os
import time
import psutil
import re
from mlx_lm import load, generate
from mlx_lm.sample_utils import make_sampler
import chromadb
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv()

class TISDEngine:
    def __init__(self, verbose=True):
        self.verbose = verbose
        self.model, self.tokenizer, self.embedder, self.collection = None, None, None, None
        self.sampler = make_sampler(temp=0.0)

    def _find_vectorstore(self):
        # We check the two places your 'ls -R' showed data
        possible_paths = [
            os.path.expanduser("~/tisd/vectorstore/chroma_db"),
            os.path.expanduser("~/tisd/vectorstore")
        ]
        for p in possible_paths:
            if os.path.exists(os.path.join(p, "chroma.sqlite3")):
                return p
        return possible_paths[0] # Fallback

    def load(self):
        t0 = time.time()
        db_path = self._find_vectorstore()
        
        if self.verbose: 
            print(f"🔍 Engine looking for data in: {db_path}")
        
        self.embedder = SentenceTransformer("all-MiniLM-L6-v2")
        client = chromadb.PersistentClient(path=db_path)
        
        # Diagnostic: What is actually in this DB?
        existing_collections = client.list_collections()
        if not existing_collections:
            raise RuntimeError(f"❌ Empty Database found at {db_path}. Run rebuild_tisd.sh again!")
            
        # Pick the collection (prioritize 'tisd_knowledge')
        names = [c.name for c in existing_collections]
        active_name = "tisd_knowledge" if "tisd_knowledge" in names else names[0]
             
        self.collection = client.get_collection(active_name)
        self.model, self.tokenizer = load("mlx-community/Phi-3-mini-4k-instruct-4bit")
        
        if self.verbose:
            print(f"✅ Engine v3.3 Ready | Collection: {active_name} | RAM: {round(psutil.virtual_memory().used / 1e9, 2)}GB")

    def clean_text(self, text):
        text = re.sub(r'<\|.*?\|>', '', text)
        text = text.replace('bys', 'by')
        text = re.sub(r'Chapter \d+\.indd \d+', '', text)
        text = re.sub(r'Reprint \d+-\d+', '', text)
        text = re.sub(r'\d{2}-\d{2}-\d{4} \d{2}:\d{2}:\d{2}', '', text)
        return re.sub(r'\s+', ' ', text).strip()

    def _retrieve(self, question, grade):
        q_emb = self.embedder.encode(question).tolist()
        res = self.collection.query(
            query_embeddings=[q_emb], 
            n_results=3, 
            where={"class_level": {"$lte": int(grade)}}
        )
        return res["documents"][0], res["metadatas"][0]

    def ask(self, question, grade=4, verbose=None):
        if verbose is None: verbose = self.verbose
        chunks, metas = self._retrieve(question, grade)
        context = " ".join([self.clean_text(c) for c in chunks])
        sources = [m.get("source", "Reference") for m in metas]
        
        messages = [
            {"role": "system", "content": "You are Tara, a teacher. Answer concisely using ONLY the facts in the context."},
            {"role": "user", "content": f"CONTEXT: {context}\\n\\nQUESTION: {question}"},
        ]
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        answer = generate(self.model, self.tokenizer, prompt=prompt, max_tokens=250, sampler=self.sampler, verbose=verbose)
        return self.clean_text(answer), list(set(sources))
"""

with open("tisd_engine_mlx.py", "w") as f:
    f.write(engine_code)

print("🚀 Engine v3.3 (Omni-Loader) saved.")


🚀 Engine v3.3 (Omni-Loader) saved.


In [13]:
# Cell 2: Initialize and Test
import importlib
import tisd_engine_mlx
importlib.reload(tisd_engine_mlx)
from tisd_engine_mlx import TISDEngine

# Load the engine
engine = TISDEngine(verbose=True)
engine.load()

# Test run
engine.ask("What is a plant?")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

✅ Engine Ready. RAM: 5.81GB
A plant is a living organism that belongs to the kingdom Plantae. It is characterized by its ability to produce its own food through the process of photosynthesis, using sunlight, water, and carbon dioxide. Plants have roots, stems, leaves, flowers, fruits, and seeds, which play essential roles in their growth, reproduction, and survival.<|end|>
Prompt: 660 tokens, 306.920 tokens-per-sec
Generation: 83 tokens, 40.965 tokens-per-sec
Peak memory: 3.230 GB


'A plant is a living organism that belongs to the kingdom Plantae. It is characterized by its ability to produce its own food through the process of photosynthesis, using sunlight, water, and carbon dioxide. Plants have roots, stems, leaves, flowers, fruits, and seeds, which play essential roles in their growth, reproduction, and survival.<|end|>'

In [14]:
# Cell 2: Test the new Engine
import sys
import importlib
import tisd_engine_mlx
importlib.reload(tisd_engine_mlx)
from tisd_engine_mlx import TISDEngine

# Initialize
engine = TISDEngine(verbose=True)
engine.load()

# Run a quick test
engine.ask("How do plants get food?")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

✅ Engine Ready. RAM: 6.27GB
Plants make their own food through a process called photosynthesis, where they use sunlight, water, and carbon dioxide to produce sugar and oxygen. The chlorophyll in their leaves traps the sunlight, which is then used to convert water and carbon dioxide into sugar, providing energy for the plant.<|end|>
Prompt: 809 tokens, 308.822 tokens-per-sec
Generation: 71 tokens, 40.094 tokens-per-sec
Peak memory: 3.230 GB


'Plants make their own food through a process called photosynthesis, where they use sunlight, water, and carbon dioxide to produce sugar and oxygen. The chlorophyll in their leaves traps the sunlight, which is then used to convert water and carbon dioxide into sugar, providing energy for the plant.<|end|>'

In [15]:
import fastapi
import uvicorn
print(f"FastAPI version: {fastapi.__version__}")
print("🚀 Localhost stack is ready for TISD!")

FastAPI version: 0.135.3
🚀 Localhost stack is ready for TISD!


In [16]:
# ~/tisd/main.py
from fastapi import FastAPI
from notebooks.tisd_engine_mlx import TISDEngine

app = FastAPI(title="TISD API")

# Initialize engine at startup
engine = TISDEngine(verbose=False)
engine.load()

@app.get("/ask")
async def ask_tara(q: str, grade: int = 4):
    answer = engine.ask(q, grade=grade)
    return {"question": q, "answer": answer}

# To run: uvicorn main:app --reload --host 0.0.0.0

ModuleNotFoundError: No module named 'notebooks'